In [72]:
import numpy as np
import matplotlib.pyplot as plt

In [73]:

# Set seed so we get the same random numbers
np.random.seed(3)
# Number of inputs
N = 3
# Number of dimensions of each input
D = 4
# Create an empty list
all_x = []
# Create elements x_n and append to list
for n in range(N):
  all_x.append(np.random.normal(size=(D,1)))
# Print out the list
print(all_x)

[array([[ 1.78862847],
       [ 0.43650985],
       [ 0.09649747],
       [-1.8634927 ]]), array([[-0.2773882 ],
       [-0.35475898],
       [-0.08274148],
       [-0.62700068]]), array([[-0.04381817],
       [-0.47721803],
       [-1.31386475],
       [ 0.88462238]])]


In [74]:
# Set seed so we get the same random numbers
np.random.seed(0)

# Choose random values for the parameters
omega_q = np.random.normal(size=(D,D))
omega_k = np.random.normal(size=(D,D))
omega_v = np.random.normal(size=(D,D))
beta_q = np.random.normal(size=(D,1))
beta_k = np.random.normal(size=(D,1))
beta_v = np.random.normal(size=(D,1))

In [75]:
# so in this cell we receive matrix where rows are embeddings

all_queries = []
all_keys = []
all_values = []

for x in all_x:
  # Standard linear transformation: Omega * x + beta
  query = np.matmul(omega_q, x) + beta_q
  key   = np.matmul(omega_k, x) + beta_k
  value = np.matmul(omega_v, x) + beta_v

  # Transpose them to (1, D) if you want to keep the rest of your loop logic
  all_queries.append(query.T)
  all_keys.append(key.T)
  all_values.append(value.T)

In [76]:
def softmax(items_in):
  
  exp_items_in = np.exp(items_in)
  
  return exp_items_in / np.sum(exp_items_in)

In [77]:
all_x_prime = []

for n in range(N):
  all_km_qn = []
  
  for keys in all_keys:
    dot_product = np.dot(all_queries[n], keys.T).item()
    all_km_qn.append(dot_product)
  
  attention = softmax(all_km_qn)
  print("Attentions for output ", n)
  print(attention)
  
  values_matrix = np.vstack(all_values)
  
  x_prime = np.matmul(attention, values_matrix)
  all_x_prime.append(x_prime)
  
# Print out true values to check you have it correct
print("x_prime_0_calculated:", all_x_prime[0])
print("x_prime_0_true: [[ 0.94744244 -0.24348429 -0.91310441 -0.44522983]]")
print("x_prime_1_calculated:", all_x_prime[1])
print("x_prime_1_true: [[ 1.64201168 -0.08470004  4.02764044  2.18690791]]")
print("x_prime_2_calculated:", all_x_prime[2])
print("x_prime_2_true: [[ 1.61949281 -0.06641533  3.96863308  2.15858316]]")
   

Attentions for output  0
[1.24326146e-13 9.98281489e-01 1.71851130e-03]
Attentions for output  1
[2.79525306e-12 5.85506360e-03 9.94144936e-01]
Attentions for output  2
[0.00505708 0.00654776 0.98839516]
x_prime_0_calculated: [ 0.94744244 -0.24348429 -0.91310441 -0.44522983]
x_prime_0_true: [[ 0.94744244 -0.24348429 -0.91310441 -0.44522983]]
x_prime_1_calculated: [ 1.64201168 -0.08470004  4.02764044  2.18690791]
x_prime_1_true: [[ 1.64201168 -0.08470004  4.02764044  2.18690791]]
x_prime_2_calculated: [ 1.61949281 -0.06641533  3.96863308  2.15858316]
x_prime_2_true: [[ 1.61949281 -0.06641533  3.96863308  2.15858316]]


In [78]:
# Define softmax operation that works independently on each column
def softmax_cols(data_in):
  # Exponentiate all of the values
  exp_values = np.exp(data_in) ;
  # Sum over columns
  denom = np.sum(exp_values, axis = 0);
  # Replicate denominator to N rows
  denom = np.matmul(np.ones((data_in.shape[0],1)), denom[np.newaxis,:])
  # Compute softmax
  softmax = exp_values / denom
  # return the answer
  return softmax

In [79]:
def self_attention(X, omega_v, omega_q, omega_k, beta_v, beta_q, beta_k):
  
  query = np.matmul(omega_q, X) + beta_q
  keys = np.matmul(omega_k, X) + beta_k
  values = np.matmul(omega_v, X) + beta_v
  
  attention = softmax_cols(keys.T @ query)
  X_prime = values @ attention
  
  return X_prime

In [80]:
# Copy data into matrix
X = np.zeros((D, N))
X[:,0] = np.squeeze(all_x[0])
X[:,1] = np.squeeze(all_x[1])
X[:,2] = np.squeeze(all_x[2])

# Run the self attention mechanism
X_prime = self_attention(X,omega_v, omega_q, omega_k, beta_v, beta_q, beta_k)

# Print out the results
print(X_prime)

[[ 0.94744244  1.64201168  1.61949281]
 [-0.24348429 -0.08470004 -0.06641533]
 [-0.91310441  4.02764044  3.96863308]
 [-0.44522983  2.18690791  2.15858316]]


In [83]:
def scaled_dot_product_self_attention(X, omega_v, omega_q, omega_k, beta_v, beta_q, beta_k):
  
  query = np.matmul(omega_q, X) + beta_q
  keys = np.matmul(omega_k, X) + beta_k
  values = np.matmul(omega_v, X) + beta_v
  D = X.shape[0]
  
  attention = softmax_cols((keys.T @ query) / np.sqrt(D))
  X_prime = values @ attention
  
  return X_prime

In [84]:
# Run the self attention mechanism
X_prime = scaled_dot_product_self_attention(X,omega_v, omega_q, omega_k, beta_v, beta_q, beta_k)

# Print out the results
print(X_prime)

[[ 0.97411966  1.59622051  1.32638014]
 [-0.23738409 -0.09516106  0.13062402]
 [-0.72333202  3.70194096  3.02371664]
 [-0.34413007  2.01339538  1.6902419 ]]
